In [1]:
import numpy as np
import pandas as pd
import cooler
from cooltools.lib.numutils import observed_over_expected, adaptive_coarsegrain
from cooltools.lib.numutils import interpolate_bad_singletons, set_diag, interp_nan
from astropy.convolution import Gaussian2DKernel
from astropy.convolution import convolve
from bioframe.io.fileops import read_bigwig

In [2]:
import torch
from pyfaidx import Fasta

In [3]:
FASTA_FILE = "/project/fudenber_735/genomes/mm10/mm10.fa"
BED_FILE = "/project/fudenber_735/tensorflow_models/akita/v2/data/mm10/sequences.bed"
COOL_FILE = "/project/fudenber_735/GEO/Hsieh2019/4DN/mESC_mm10_4DNFILZ1CPT8.mapq_30.2048.cool"
FOLD = 0

# --- Load Data ---
genome = Fasta(FASTA_FILE)
df = pd.read_csv(BED_FILE, sep="\t", header=None, names=["chrom", "start", "end", "fold"])
df_select_fold = df[df["fold"] == f"fold{FOLD}"].reset_index(drop=True)

genome_hic_cool = cooler.Cooler(COOL_FILE)

In [4]:
import random

def one_hot_encode_sequence(sequence_obj):
    sequence = str(sequence_obj).upper()
    base_to_int = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

    encoded_sequence = np.array([
        base_to_int.get(base, base_to_int[random.choice("ACGT")]) for base in sequence
    ])

    one_hot_encoded = np.zeros((4, len(encoded_sequence)), dtype=np.float32)
    one_hot_encoded[encoded_sequence, np.arange(len(encoded_sequence))] = 1

    return np.expand_dims(one_hot_encoded, axis=0)

In [5]:
def process_hic_matrix(genome_hic_cool, mseq_str, diagonal_offset=2, padding=64, kernel_stddev=1):
    seq_hic_raw = genome_hic_cool.matrix(balance=True).fetch(mseq_str)
    
    # Check for NaN filtering percentage
    seq_hic_nan = np.isnan(seq_hic_raw)
    num_filtered_bins = np.sum(np.sum(seq_hic_nan, axis=0) == len(seq_hic_nan))
    print("num_filtered_bins:", num_filtered_bins)
    
    if num_filtered_bins > (0.5 * len(seq_hic_nan)):
        print(f"More than 50% bins filtered in {mseq_str}. Check Hi-C data quality.")
        
    # clip first diagonals and high values
    clipval = np.nanmedian(np.diag(seq_hic_raw, diagonal_offset))
    for i in range(-diagonal_offset+1, diagonal_offset):
        set_diag(seq_hic_raw, clipval, i)
    seq_hic_raw = np.clip(seq_hic_raw, 0, clipval)
    seq_hic_raw[seq_hic_nan] = np.nan
    
    # adaptively coarsegrain based on raw counts
    seq_hic_smoothed = adaptive_coarsegrain(
                            seq_hic_raw,
                            genome_hic_cool.matrix(balance=False).fetch(mseq_str),
                            cutoff=2, max_levels=8)
    seq_hic_nan = np.isnan(seq_hic_smoothed)
    
    # local obs/exp
    seq_hic_obsexp = observed_over_expected(seq_hic_smoothed, ~seq_hic_nan)[0]
    
    log_hic_obsexp = np.log(seq_hic_obsexp)
    
    # Apply padding
    if padding > 0:
        log_hic_obsexp = log_hic_obsexp[padding:-padding, padding:-padding]
    
    log_hic_obsexp = interp_nan(log_hic_obsexp)
    for i in range(-diagonal_offset+1, diagonal_offset): set_diag(log_hic_obsexp, 0,i)
    
    kernel = Gaussian2DKernel(x_stddev=kernel_stddev)
    seq_hic = convolve(log_hic_obsexp, kernel)
    
    return seq_hic    

In [6]:
def upper_triangular_to_vector_skip_diagonals(matrix, dim=512, diag=2):
    
    # Extract the upper triangular part excluding the first two diagonals
    upper_triangular_vector = matrix[np.triu_indices(dim, k=diag)]
    
    return upper_triangular_vector

In [7]:
N = 256
diagonal_offset = 2

In [8]:
out_dir = "/scratch1/smaruj/alpha_genome_validation/fold0"

In [9]:
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
def plot_map(matrix, vmin=-0.6, vmax=0.6, palette="RdBu_r", width=5, height=5):
    fig, axes = plt.subplots(1, 1, figsize=(width, height))

    sns.heatmap(
        matrix,
        vmin=vmin,
        vmax=vmax,
        cbar=False,
        cmap=palette,
        square=True,
        xticklabels=False,
        yticklabels=False,
        ax=axes
    )

    plt.tight_layout()
    plt.show()

In [11]:
N = 256
diagonal_offset = 2

In [12]:
def vector_to_symmetric_matrix(vec, N):
    matrix = torch.zeros((N, N), dtype=vec.dtype)
    triu_indices = torch.triu_indices(N, N)
    matrix[triu_indices[0], triu_indices[1]] = vec
    matrix = matrix + matrix.T - torch.diag(torch.diag(matrix))
    return matrix

In [13]:
import scipy.stats

In [14]:
# Exclude diagonals: 0 and ±1
def get_upper_tri_mask(n, skip_diagonals=2):
    # Create mask with False on excluded diagonals, True elsewhere in upper triangle
    mask = np.triu(np.ones((n, n), dtype=bool), k=skip_diagonals)
    return mask

In [15]:
df_select_fold

,chrom,start,end,fold
0,chr5,63203328,64514048,fold0
1,chr3,138672128,139982848,fold0
2,chr5,43542528,44853248,fold0
3,chr3,115171328,116482048,fold0
4,chr7,61700096,63010816,fold0
...,...,...,...,...
720,chr5,150165504,151476224,fold0
721,chr11,66576384,67887104,fold0
722,chrX,57774080,59084800,fold0
723,chr11,67559424,68870144,fold0


In [16]:
df_select_fold["ag_start"] = df_select_fold["start"] + 64*2048
df_select_fold["ag_end"] = df_select_fold["end"] - 64*2048

In [17]:
pearson_r = []
spearman_r = []
mse_list = []

for i, row in enumerate(df_select_fold.itertuples(index=False)):
    print("index:", i)
    chrom, start, end = row.chrom, row.start, row.end
    mseq_str = f"{chrom}:{start}-{end}"
    
    # TARGET
    sequence = genome[chrom][start:end]
    ohe_sequence = one_hot_encode_sequence(sequence)
    matrix = process_hic_matrix(genome_hic_cool, mseq_str, diagonal_offset=2, padding=64, kernel_stddev=1)
    
    # plot_map(matrix)
    
    # ORCA'S PRED
    filename = f"{chrom}_{row.ag_start}_{row.ag_end}.npy"
    file_path = f"{out_dir}/{filename}"
    
    ag_matrix = np.load(file_path)
    
    for i in range(-1, 2):
        diag_indices = np.diag_indices_from(ag_matrix)
        if i < 0:
            ag_matrix[diag_indices[0][:i], diag_indices[1][-i:]] = np.nan
        elif i > 0:
            ag_matrix[diag_indices[0][i:], diag_indices[1][:-i]] = np.nan
        else:
            ag_matrix[diag_indices] = np.nan
    
    # plot_map(ag_matrix)
    
    mask = get_upper_tri_mask(matrix.shape[0])
    target_vec = matrix[mask]
    orca_vec = ag_matrix[mask]
    
    # Remove NaNs (e.g., from diagonals)
    valid_mask = ~np.isnan(target_vec) & ~np.isnan(orca_vec)
    target_vec = target_vec[valid_mask]
    orca_vec = orca_vec[valid_mask]
    
    # Metrics
    r, _ = scipy.stats.pearsonr(target_vec, orca_vec)
    rho, _ = scipy.stats.spearmanr(target_vec, orca_vec)
    mse = np.mean((target_vec - orca_vec) ** 2)
    
    # Save
    pearson_r.append(r)
    spearman_r.append(rho)
    mse_list.append(mse)

index: 0
num_filtered_bins: 20


/home1/smaruj/miniconda3/envs/pytorch_cuda11.8/lib/python3.12/site-packages/cooltools/lib/numutils.py:1376: RuntimeWarning: invalid value encountered in divide
  val_cur = ar_cur / armask_cur


index: 1
num_filtered_bins: 14


/home1/smaruj/miniconda3/envs/pytorch_cuda11.8/lib/python3.12/site-packages/cooltools/lib/numutils.py:1376: RuntimeWarning: invalid value encountered in divide
  val_cur = ar_cur / armask_cur


index: 2
num_filtered_bins: 10
index: 3
num_filtered_bins: 13
index: 4
num_filtered_bins: 28
index: 5
num_filtered_bins: 2
index: 6
num_filtered_bins: 21
index: 7
num_filtered_bins: 41
index: 8
num_filtered_bins: 21
index: 9
num_filtered_bins: 1
index: 10
num_filtered_bins: 4
index: 11
num_filtered_bins: 10
index: 12
num_filtered_bins: 27
index: 13
num_filtered_bins: 21
index: 14
num_filtered_bins: 21
index: 15
num_filtered_bins: 0
index: 16
num_filtered_bins: 9
index: 17
num_filtered_bins: 27
index: 18
num_filtered_bins: 8
index: 19
num_filtered_bins: 11
index: 20
num_filtered_bins: 24
index: 21
num_filtered_bins: 32
index: 22
num_filtered_bins: 25
index: 23
num_filtered_bins: 14
index: 24
num_filtered_bins: 22
index: 25
num_filtered_bins: 9
index: 26
num_filtered_bins: 28
index: 27
num_filtered_bins: 11
index: 28
num_filtered_bins: 24
index: 29
num_filtered_bins: 6
index: 30
num_filtered_bins: 28
index: 31
num_filtered_bins: 24
index: 32
num_filtered_bins: 19
index: 33
num_filtered_b

In [18]:
print("Average PearsonR:", np.mean(pearson_r))

Average PearsonR: 0.5315088181125873


In [19]:
print("Average SpearmanR:", np.mean(spearman_r))
print("Average MSE:", np.mean(mse_list))

Average SpearmanR: 0.5185985552142169
Average MSE: 0.14056128357713044
